# Course 1 — The Numerical Engine (NumPy)

Live-coding notebook that mirrors the slide deck.
Concrete examples lifted from Aurélien Géron's NumPy tutorial.

**Sections**
1. The ND-Array (0:00–0:30)
2. Shape & Slicing (0:30–1:00)
3. Vectorize (1:00–1:30)

In [ ]:
import numpy as np
np.__version__


## 1. The ND-Array

### List vs. ndarray — same data, different machine

In [ ]:
py_list = [1, 2, 3, 4]
arr = np.array([1, 2, 3, 4])
print(py_list, type(py_list))
print(arr, type(arr), arr.dtype)


### Measure it

In [ ]:
%timeit sum(range(1_000_000))
%timeit np.arange(1_000_000).sum()


### Creating arrays — the recipes you'll use weekly

In [ ]:
print(np.zeros(5))
print(np.zeros((3, 4)))
print(np.ones((3, 4)))
print(np.full((3, 4), np.pi))


In [ ]:
print(np.arange(1, 5))
print(np.arange(1, 5, 0.5))
print(np.linspace(0, 5/3, 6))
print(np.eye(3))


In [ ]:
rng = np.random.default_rng(seed=42)
print(rng.random((3, 4)))
print(rng.standard_normal((3, 4)))


### dtypes — promotion in action

In [ ]:
c = np.arange(1, 5)
print(c.dtype, c)        # int64

k1 = np.arange(0, 5, dtype=np.uint8)
k2 = k1 + np.array([5, 6, 7, 8, 9], dtype=np.int8)
print(k2.dtype, k2)      # int16

k3 = k1 + 1.5
print(k3.dtype, k3)      # float64


### Inspect any array in six attributes

In [ ]:
a = np.zeros((3, 4))
print(a.shape, a.ndim, a.size, a.itemsize, a.nbytes, a.dtype)


## 2. Shape & Slicing

### Reshape, ravel, transpose

In [ ]:
g = np.arange(24)
g.shape = (6, 4)            # reshape in place
print(g)

g2 = g.reshape(4, 6)        # new view, same buffer
print(g2)

print(g.ravel())            # back to 1-D


In [ ]:
t = np.arange(24).reshape(4, 2, 3)
print(t.transpose((1, 2, 0)).shape)   # (2, 3, 4)

m1 = np.arange(10).reshape(2, 5)
print(m1.T)


### 2-D slicing — the main move

In [ ]:
b = np.arange(48).reshape(4, 12)
print(b)
print('b[1, 2]   =', b[1, 2])
print('b[1, :]   =', b[1, :])
print('b[:, 1]   =', b[:, 1])
print('b[1:3,2:5]=\n', b[1:3, 2:5])
print('b[::-1]   =\n', b[::-1])


### Fancy indexing — pick whatever rows/cols you want

In [ ]:
print(b[(0, 2), 2:5])                       # rows 0 and 2, cols 2-4
print(b[:, (-1, 2, -1)])                     # cols last, 2, last (repeats!)
print(b[(-1, 2, -1, 2), (5, 9, 1, 9)])       # paired indices — 4 cells


### Boolean indexing

In [ ]:
m = np.array([20, -5, 30, 40])
print(m[m < 25])             # [20 -5]

rows_on = np.array([True, False, True, False])
print(b[rows_on, :])         # keep rows 0 and 2

print(b[b % 3 == 1])         # every elem where x%3 == 1


### Gotcha: slices are *views*

In [ ]:
a = np.arange(12).reshape(3, 4)
view = a[:2, :2]
view[0, 0] = -999
print(a)        # the source mutated too!


## 3. Vectorize

### The loop you stop writing

In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(size=1_000_000)

def slow(x):
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = x[i]**2 + 3*x[i]
    return out

def fast(x):
    return x**2 + 3*x

%timeit slow(x)
%timeit fast(x)


### Element-wise arithmetic on paired arrays

In [ ]:
a = np.array([14, 23, 32, 41])
b = np.array([5,  4,  3,  2])
print(a + b)
print(a - b)
print(a * b)
print(a / b)
print(np.maximum(a, b))
print(np.greater(a, b))


### Universal functions — math everywhere

In [ ]:
print(np.sqrt(a))
print(np.exp(a))
print(np.sin(a))
print(np.maximum(a, 0))     # ReLU in one line
print(np.clip(a, 0, 30))


### Reductions — and the axis mnemonic

In [ ]:
a = np.array([[-2.5, 3.1, 7], [10, 11, 12]])
print('mean all   :', a.mean())

c = np.arange(24).reshape(2, 3, 4)
print('sum axis 0 :', c.sum(axis=0).shape)   # collapse axis 0 -> (3, 4)
print('sum axis 1 :', c.sum(axis=1).shape)   # collapse axis 1 -> (2, 4)
print('sum (0, 2) :', c.sum(axis=(0, 2)).shape)


### Broadcasting — three live examples

In [ ]:
k = np.arange(6).reshape(2, 3)
print(k)

# (2,3) + (3,) -> broadcast row across both rows
print(k + [100, 200, 300])

# (2,3) + (2,1) -> broadcast column across cols
print(k + [[100], [200]])

# (1,1,5) + (5,) -> still (1,1,5)
h = np.arange(5).reshape(1, 1, 5)
print(h + [10, 20, 30, 40, 50])


### Matrix multiplication

In [ ]:
n1 = np.arange(10).reshape(2, 5)
n2 = np.arange(15).reshape(5, 3)
print(n1.dot(n2))   # equivalent to n1 @ n2


### Linear algebra

In [ ]:
import numpy.linalg as linalg

m = np.array([[1,  2,  3],
              [5,  7, 11],
              [21, 29, 31]])
print('inv:\n', linalg.inv(m))
print('det:', linalg.det(m))

A = np.array([[2, 6], [5, 3]])
b_vec = np.array([6, -9])
print('solve Ax=b:', linalg.solve(A, b_vec))   # [-3.  2.]


### Worked example — standardize a data matrix

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(loc=5, scale=2, size=(200, 5))
Z = (X - X.mean(axis=0)) / X.std(axis=0)
print('column means ~0:', Z.mean(axis=0).round(4))
print('column stds  ~1:', Z.std(axis=0).round(4))


### Recap
* ndarray is a contiguous typed buffer — that is the entire speed story.
* Reshape and transpose change *strides*, not data.
* Replace numeric `for` loops with broadcasting + reductions.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

# Exercise 1 — Array Basics

Goal: get fluent with creation, dtypes, and the shape vocabulary.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)


**Task 1.** Create a `(4, 5)` array of zeros with dtype `int32`. Print its shape, dtype, and total memory in bytes.

In [ ]:
# your code here


**Task 2.** Build a 1-D array of 50 evenly-spaced values between -3 and 3 inclusive.

In [ ]:
# your code here


**Task 3.** Draw 10000 standard-normal samples. Compute the empirical mean and std and check they are close to 0 and 1.

In [ ]:
# your code here


**Task 4.** Make a `(3, 3)` identity matrix without using `np.eye`. (Hint: `np.arange` + boolean comparison.)

In [ ]:
# your code here


### Exercise 1 — Solution

# Exercise 1 — Solutions

In [ ]:
import numpy as np
rng = np.random.default_rng(42)


**Task 1.**

In [ ]:
a = np.zeros((4, 5), dtype=np.int32)
print(a.shape, a.dtype, a.nbytes, 'bytes')


**Task 2.**

In [ ]:
xs = np.linspace(-3, 3, 50)
print(xs[:5], '...', xs[-5:])
print('count:', xs.size)


**Task 3.**

In [ ]:
samples = rng.normal(size=10_000)
print('mean:', samples.mean(), '  std:', samples.std())
assert abs(samples.mean()) < 0.05
assert abs(samples.std() - 1) < 0.05


**Task 4.**

In [ ]:
n = 3
ix = np.arange(n)
I = (ix[:, None] == ix[None, :]).astype(int)
print(I)
assert np.array_equal(I, np.eye(n, dtype=int))


---

## Exercise 2

# Exercise 2 — Slicing & Reshape

We will treat an image as an ndarray — exactly what every CV model sees.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# Synthetic 'image': a 64x64 grayscale gradient with a bright square in the center.
img = np.tile(np.linspace(0, 1, 64), (64, 1))
img[24:40, 24:40] = 1.0
plt.imshow(img, cmap='gray'); plt.title('source'); plt.axis('off');


**Task 1.** Crop the central 32×32 region of `img` and display it.

In [ ]:
# your code here


**Task 2.** Flip the image left↔right and top↔bottom (two separate outputs).

In [ ]:
# your code here


**Task 3.** Convert the image to RGB by stacking it 3 times along a new last axis. Print the new shape.

In [ ]:
# your code here


**Task 4.** Set every pixel whose value is above 0.9 to 0 (in a copy, not the original).

In [ ]:
# your code here


### Exercise 2 — Solution

# Exercise 2 — Solutions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
img = np.tile(np.linspace(0, 1, 64), (64, 1))
img[24:40, 24:40] = 1.0


**Task 1.** Center crop.

In [ ]:
crop = img[16:48, 16:48]
print(crop.shape)
plt.imshow(crop, cmap='gray'); plt.axis('off');


**Task 2.** Flips.

In [ ]:
lr = img[:, ::-1]
ud = img[::-1, :]
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(lr, cmap='gray'); ax[0].set_title('LR'); ax[0].axis('off')
ax[1].imshow(ud, cmap='gray'); ax[1].set_title('UD'); ax[1].axis('off');


**Task 3.** Grayscale -> RGB.

In [ ]:
rgb = np.stack([img, img, img], axis=-1)
print(rgb.shape)        # (64, 64, 3)


**Task 4.** Threshold on a copy.

In [ ]:
out = img.copy()
out[out > 0.9] = 0
print('max before:', img.max(), 'max after:', out.max())


---

## Exercise 3

# Exercise 3 — Vectorization & Broadcasting

Replace every `for` loop with a vectorized expression.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = rng.normal(size=1_000_000)


**Task 1.** Compute the per-element sigmoid `1 / (1 + exp(-x))` for `x` — no loops.

In [ ]:
# your code here


**Task 2.** Standardize a `(200, 5)` data matrix column-wise (subtract column mean, divide by column std).

In [ ]:
data = rng.normal(loc=5, scale=2, size=(200, 5))
# your code here


**Task 3.** Given 50 2-D points in `pts`, compute the full 50×50 pairwise Euclidean distance matrix. No loops.

In [ ]:
pts = rng.normal(size=(50, 2))
# your code here


**Task 4 (stretch).** Time a Python-loop version of Task 3 vs your vectorized version with `%timeit`.

In [ ]:
# your code here


### Exercise 3 — Solution

# Exercise 3 — Solutions

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = rng.normal(size=1_000_000)


**Task 1.**

In [ ]:
sig = 1 / (1 + np.exp(-x))
print(sig[:5])


**Task 2.** Column-wise standardize.

In [ ]:
data = rng.normal(loc=5, scale=2, size=(200, 5))
z = (data - data.mean(axis=0)) / data.std(axis=0)
print('column means ~0:', z.mean(axis=0).round(4))
print('column stds  ~1:', z.std(axis=0).round(4))


**Task 3.** Pairwise distances.

In [ ]:
pts = rng.normal(size=(50, 2))
diff = pts[:, None, :] - pts[None, :, :]
dist = np.sqrt((diff**2).sum(axis=-1))
print(dist.shape)
assert np.allclose(dist, dist.T)
assert np.allclose(np.diag(dist), 0)


**Task 4.** Timing showdown.

In [ ]:
def naive(pts):
    n = len(pts)
    out = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            out[i, j] = np.sqrt(((pts[i] - pts[j])**2).sum())
    return out

def fast(pts):
    diff = pts[:, None, :] - pts[None, :, :]
    return np.sqrt((diff**2).sum(axis=-1))

%timeit naive(pts)
%timeit fast(pts)
